# 05 — Retrieval evaluation and hybrid search

Stream C, tasks **C4–C5**. This notebook compares two dense encoders on a fixed, stratified 5% sample of LEDGAR test clauses, then evaluates BM25 and reciprocal-rank fusion (RRF) on the same sample. The automatic benchmark defines relevance as **same LEDGAR label** and uses the full 60,000-clause training split as the retrieval bank. The drafted lawyer questions are not evaluated here because no domain-expert relevance judgments are available.

The fixed seed and per-label sampling make this benchmark reproducible while reducing query-time evaluation. Indexing still embeds the full training bank for each model. A GPU can speed up embedding; all index embeddings are cached under the gitignored `data/indexes/` directory.

## 1. Load the fixed train bank and test queries

In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # If the notebook runs from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

In [14]:
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import RESULTS_DIR
from src.data import load_ledgar, to_pandas
from src.retrieval.dense import MODEL_NAME, build_dense_index
from src.retrieval.evaluation import evaluate_same_label_with_bank
from src.retrieval.hybrid import BM25Retriever, HybridRetriever
from src.utils import set_seed

set_seed(42)
dataset = load_ledgar()
train_df = to_pandas(dataset, "train")
test_df = to_pandas(dataset, "test")
bank_labels = dict(zip(train_df["idx"].astype(int), train_df["label"].astype(int)))

EVAL_FRACTION = 0.05
retrieval_test_df = pd.concat(
    [
        group.sample(
            n=min(len(group), max(1, round(len(group) * EVAL_FRACTION))),
            random_state=42,
        )
        for _, group in test_df.groupby("label", sort=True)
    ]
).sort_values("idx").reset_index(drop=True)

print(
    f"Bank: {len(train_df):,} training clauses | Retrieval queries: "
    f"{len(retrieval_test_df):,} / {len(test_df):,} "
    f"({len(retrieval_test_df) / len(test_df):.1%}; stratified by label)"
)


Bank: 60,000 training clauses | Retrieval queries: 499 / 10,000 (5.0%; stratified by label)


## 2. Compare two sentence-transformer encoders (C4)

For each query in the fixed, stratified 5% test sample, retrieve the top 10 clauses from the full training bank. A result is relevant iff its label equals the query clause's label. P@5 divides by five; MRR is computed over the returned top-10; nDCG uses binary relevance at 10. This is a sampled label-retrieval benchmark, not a judgment that the retrieved text answers every semantic detail.

In [15]:
encoder_names = [
    MODEL_NAME,
    "BAAI/bge-small-en-v1.5",
]

dense_retrievers = {}
automatic_rows = []
for model_name in encoder_names:
    print(f"\nBuilding/loading index: {model_name}")
    retriever = build_dense_index(train_df, model_name=model_name, batch_size=64)
    dense_retrievers[model_name] = retriever
    started = time.perf_counter()
    rankings = retriever.search_many(retrieval_test_df["text"].tolist(), k=10)
    elapsed_s = time.perf_counter() - started
    metrics = evaluate_same_label_with_bank(
        retrieval_test_df["label"].tolist(), rankings, bank_labels, precision_k=5, ndcg_k=10
    )
    automatic_rows.append({
        "method": model_name,
        "query_set": "Stratified 5% LEDGAR test; relevance=same label",
        **metrics,
        "search_time_s": round(elapsed_s, 2),
        "index_cache_hit": retriever.cache_hit,
    })
    del rankings

dense_results_df = pd.DataFrame(automatic_rows)
display(dense_results_df)


Building/loading index: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Building/loading index: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

,method,query_set,n_queries,precision@5,mrr@10,ndcg@10,search_time_s,index_cache_hit
0,sentence-transformers/all-MiniLM-L6-v2,Stratified 5% LEDGAR test; relevance=same label,499,0.763928,0.851221,0.742074,14.57,True
1,BAAI/bge-small-en-v1.5,Stratified 5% LEDGAR test; relevance=same label,499,0.787575,0.875221,0.772348,15.06,True


## 3. Evaluate BM25 and dense + BM25 RRF (C5)

The lexical baseline is BM25 over the same training bank. Hybrid search fuses dense and BM25 ranks using RRF with constant 60. The optional `predicted_label` argument filters results to a classifier-predicted class; this notebook evaluates the unfiltered form so retrieval quality is not conflated with classifier errors.

In [16]:
mini_retriever = dense_retrievers[MODEL_NAME]
bm25_retriever = BM25Retriever(train_df)
hybrid_retriever = HybridRetriever(mini_retriever, bm25_retriever)

test_queries = retrieval_test_df["text"].tolist()
for method_name, retrieve_many in [
    ("BM25", lambda: bm25_retriever.search_many(test_queries, k=10)),
    ("MiniLM + BM25 (RRF, c=60)", lambda: hybrid_retriever.search_many(test_queries, k=10)),
]:
    print(f"Evaluating {method_name} on {len(test_queries):,} sampled test queries")
    started = time.perf_counter()
    rankings = retrieve_many()
    elapsed_s = time.perf_counter() - started
    metrics = evaluate_same_label_with_bank(
        retrieval_test_df["label"].tolist(), rankings, bank_labels, precision_k=5, ndcg_k=10
    )
    automatic_rows.append({
        "method": method_name,
        "query_set": "Stratified 5% LEDGAR test; relevance=same label",
        **metrics,
        "search_time_s": round(elapsed_s, 2),
        "index_cache_hit": None,
    })
    del rankings

automatic_results_df = pd.DataFrame(automatic_rows)
display(automatic_results_df)

Evaluating BM25 on 499 sampled test queries
Evaluating MiniLM + BM25 (RRF, c=60) on 499 sampled test queries


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

,method,query_set,n_queries,precision@5,mrr@10,ndcg@10,search_time_s,index_cache_hit
0,sentence-transformers/all-MiniLM-L6-v2,Stratified 5% LEDGAR test; relevance=same label,499,0.763928,0.851221,0.742074,14.57,True
1,BAAI/bge-small-en-v1.5,Stratified 5% LEDGAR test; relevance=same label,499,0.787575,0.875221,0.772348,15.06,True
2,BM25,Stratified 5% LEDGAR test; relevance=same label,499,0.779960,0.868339,0.755349,624.41,None
3,"MiniLM + BM25 (RRF, c=60)",Stratified 5% LEDGAR test; relevance=same label,499,0.784369,0.868461,0.766485,645.95,None


## 4. Save the automatic comparison

This writes the versioned summary table. The held-out test split is used here only for the pre-specified sampled retrieval benchmark; changing model or index decisions based on these results should be documented.

In [17]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary_path = RESULTS_DIR / "retrieval_summary.csv"
automatic_results_df.to_csv(summary_path, index=False)
print(f"Saved {len(automatic_results_df)} retrieval rows to {summary_path}")

Saved 4 retrieval rows to /Users/brianooi/Desktop/Albert/Deep Learning/Projet_Deep_Learning/results/retrieval_summary.csv
